In [1]:
import time
import numpy as np
import pandas as pd
import openml
from pymfe.mfe import MFE
import warnings

suite = openml.study.get_suite("OpenML-CC18")
tasks = openml.tasks.list_tasks(task_id=suite.tasks, output_format="dataframe")
print(len(tasks))
print(tasks.columns.tolist())

72
['tid', 'ttid', 'did', 'name', 'task_type', 'status', 'estimation_procedure', 'source_data', 'target_feature', 'MajorityClassSize', 'MaxNominalAttDistinctValues', 'MinorityClassSize', 'NumberOfClasses', 'NumberOfFeatures', 'NumberOfInstances', 'NumberOfInstancesWithMissingValues', 'NumberOfMissingValues', 'NumberOfNumericFeatures', 'NumberOfSymbolicFeatures']


In [2]:
print(tasks.estimation_procedure.value_counts())

selected_tasks = tasks[
    tasks.NumberOfInstances.between(500, 50000) &
    (tasks.NumberOfFeatures <= 500)
]

print(len(selected_tasks))

tam = selected_tasks.assign(tam=selected_tasks.NumberOfInstances * selected_tasks.NumberOfFeatures).sort_values("tam")
amostra = tam.iloc[[0, len(tam) // 2, -1]]

def pegar_dados(did, alvo, tentativas=3):
    for k in range(tentativas):
        try:
            ds = openml.datasets.get_dataset(int(did))
            return ds.get_data(target=alvo, dataset_format="dataframe")
        except Exception as e:
            print("   falha ao baixar, tentando de novo:", repr(e)[:120])
            time.sleep(10)
    raise RuntimeError("não consegui baixar o dataset")

def testar_pymfe(linha):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i in cat_cols:
        col = X.columns[i]
        cod = X[col].astype("category").cat.codes.astype(float)
        X[col] = cod.where(cod >= 0)
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    X_arr = X.to_numpy(dtype=float)

    t0 = time.time()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X_arr, y_arr, cat_cols=cat_cols)
    nomes, valores = mfe.extract()
    return len(nomes), time.time() - t0

for linha in amostra.itertuples():
    print(linha.name, "|", linha.NumberOfInstances, "instâncias |", linha.NumberOfFeatures, "features")
    try:
        n, seg = testar_pymfe(linha)
        print("   ", n, "metacaracterísticas em", f"{seg:.1f}s")
    except Exception as e:
        print("    ERRO:", repr(e)[:300])

estimation_procedure
10-fold Crossvalidation    72
Name: count, dtype: int64
60
balance-scale | 625 instâncias | 5 features


c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\_internal.py:1567: UserWarning: It is not possible make equal discretization
  warnings.warn("It is not possible make equal discretization")
c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\_internal.py:1567: UserWarning: It is not possible make equal discretization
  warnings.warn("It is not possible make equal discretization")


    73 metacaracterísticas em 0.2s
analcatdata_authorship | 841 instâncias | 71 features


c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\statistical.py:1266: RuntimeWarning: invalid value encountered in log
  np.log(np.linalg.det(S_i)) for S_i in sample_cov_matrices
c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\statistical.py:1271: RuntimeWarning: invalid value encountered in log
  * np.log(np.linalg.det(pooled_cov_mat))


    73 metacaracterísticas em 1.2s
nomao | 34465 instâncias | 119 features


c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\_internal.py:1567: UserWarning: It is not possible make equal discretization
  warnings.warn("It is not possible make equal discretization")
c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\_internal.py:732: RuntimeWarning: Can't summarize feature 'can_cor' with summary 'sd'. Will set it as 'np.nan'.
  warnings.warn(
c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\scipy\stats\_morestats.py:2199: RuntimeWarning: invalid value encountered in scalar divide
  return num / den
c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\statistical.py:1266: RuntimeWarning: divide by zero encountered in log
  np.log(np.linalg.det(S_i)) for S_i in sample_cov_matrices
c:\Users\caioc\recomendacao-classificadores\.venv\Lib\site-packages\pymfe\statistical.py:1271: RuntimeWarning: divide by zero encountered in log
  * np.log(np.linalg.det(pooled_cov_mat))
c:\Users\cai

    73 metacaracterísticas em 8.6s


In [3]:
rng = np.random.default_rng(0)

def mf_com_codigos(linha, permutar):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i in cat_cols:
        col = X.columns[i]
        cod = X[col].astype("category").cat.codes.to_numpy()
        if permutar:
            perm = rng.permutation(cod.max() + 1)
            cod = np.where(cod >= 0, perm[np.clip(cod, 0, None)], -1)
        X[col] = np.where(cod >= 0, cod, np.nan).astype(float)
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()

    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=float), y_arr, cat_cols=cat_cols)
    nomes, valores = mfe.extract()
    return pd.Series(valores, index=nomes, dtype=float)

linha = next(selected_tasks[selected_tasks.name == "car"].itertuples())
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    a  = mf_com_codigos(linha, permutar=False)
    a2 = mf_com_codigos(linha, permutar=False)
    b  = mf_com_codigos(linha, permutar=True)

def mudou(x, z):
    return (x.isna() != z.isna()) | ((x - z).abs() > 1e-9)

print("controle (a vs a2):", mudou(a, a2).sum(), "de", len(a), "mudaram")
print("embaralhado (a vs b):", mudou(a, b).sum(), "de", len(a), "mudaram")
print(a[mudou(a, b)].index.tolist())

controle (a vs a2): 0 de 73 mudaram
embaralhado (a vs b): 1 de 73 mudaram
['gravity']


In [4]:
def mf_freq(linha):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i in cat_cols:
        col = X.columns[i]
        freq = X[col].value_counts(normalize=True).to_dict()
        X[col] = X[col].astype(object).map(freq).astype(float)
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=float), y_arr, cat_cols=cat_cols)
    nomes, valores = mfe.extract()
    return pd.Series(valores, index=nomes, dtype=float)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    f = mf_freq(linha)

dif = mudou(a, f)
print(dif.sum(), "de", len(a), "diferem da codificação ordinal")
print(a[dif].index.tolist())

56 de 73 diferem da codificação ordinal
['attr_conc.mean', 'attr_conc.sd', 'attr_ent.mean', 'attr_ent.sd', 'can_cor.mean', 'can_cor.sd', 'class_conc.mean', 'class_conc.sd', 'cor.mean', 'cor.sd', 'cov.mean', 'cov.sd', 'eigenvalues.mean', 'eigenvalues.sd', 'eq_num_attr', 'g_mean.mean', 'g_mean.sd', 'gravity', 'h_mean.mean', 'h_mean.sd', 'iq_range.mean', 'iq_range.sd', 'joint_ent.mean', 'joint_ent.sd', 'kurtosis.mean', 'kurtosis.sd', 'lh_trace', 'mad.sd', 'max.sd', 'mean.mean', 'mean.sd', 'median.mean', 'median.sd', 'min.mean', 'min.sd', 'mut_inf.mean', 'mut_inf.sd', 'nr_cor_attr', 'nr_disc', 'nr_outliers', 'ns_ratio', 'p_trace', 'range.mean', 'range.sd', 'roy_root', 'sd.mean', 'sd.sd', 'skewness.mean', 'skewness.sd', 'sparsity.mean', 'sparsity.sd', 't_mean.mean', 't_mean.sd', 'var.mean', 'var.sd', 'w_lambda']


In [5]:
print(selected_tasks.loc[selected_tasks.name == "car", "NumberOfInstances"].iloc[0], "=", 4*4*4*3*3*3)
X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
print(X.iloc[:, 0].value_counts())

1728 = 1728
buying
vhigh    432
high     432
med      432
low      432
Name: count, dtype: int64


In [6]:
cand = selected_tasks[selected_tasks.name == "credit-g"]
print(len(cand), "base(s) encontrada(s)")
linha2 = next(cand.itertuples())

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    a_  = mf_com_codigos(linha2, permutar=False)
    a2_ = mf_com_codigos(linha2, permutar=False)
    b_  = mf_com_codigos(linha2, permutar=True)
    f_  = mf_freq(linha2)

print("controle      :", mudou(a_, a2_).sum(), "de", len(a_))
print("embaralhado   :", mudou(a_, b_).sum(), "de", len(a_))
print("frequência    :", mudou(a_, f_).sum(), "de", len(a_))
print(a_[mudou(a_, b_)].index.tolist())

1 base(s) encontrada(s)
controle      : 2 de 73
embaralhado   : 27 de 73
frequência    : 56 de 73
['attr_conc.mean', 'attr_conc.sd', 'cor.mean', 'cor.sd', 'cov.mean', 'cov.sd', 'eigenvalues.mean', 'eigenvalues.sd', 'gravity', 'iq_range.mean', 'iq_range.sd', 'kurtosis.mean', 'kurtosis.sd', 'mean.mean', 'mean.sd', 'median.mean', 'median.sd', 'nr_cor_attr', 'nr_outliers', 'sd.mean', 'sd.sd', 'skewness.mean', 'skewness.sd', 't_mean.mean', 't_mean.sd', 'var.mean', 'var.sd']


In [7]:
print(MFE.fit.__doc__)

Fits dataset into an MFE model.

Parameters
----------
X : :obj:`List`
    Predictive attributes of the dataset.

y : :obj:`List`, optional
    Target attributes of the dataset, assuming that it is a supervised
    task.

transform_num : :obj:`bool`, optional
    If True, numeric attributes are discretized using equal-frequency
    histogram technique to use alongside categorical data when
    extracting categoric-only metafeatures. Note that numeric-only
    features still uses the original numeric values, not the
    discretized ones. If False, then numeric attributes are ignored for
    categorical-only meta-features.

transform_cat : :obj:`str`, optional
    Transform categorical data to use alongside numerical data while
    extracting numeric-only metafeatures. Note that categoric-only
    features still uses the original categoric values, and not the
    binarized ones.

    If `one-hot`, categorical attributes are binarized using one-hot
    encoding with `k-1` features for a c

In [8]:
def mf_nativo(linha, transform_cat="gray", renomear=False):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i in cat_cols:
        col = X.columns[i]
        s = X[col].astype(object)
        if renomear:
            cats = list(s.dropna().unique())
            novo = dict(zip(cats, rng.permutation(len(cats))))
            s = s.map(lambda v: f"c{novo[v]}" if v in novo else v)
        X[col] = s
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=object), y_arr, cat_cols=cat_cols, transform_cat=transform_cat)
    nomes, valores = mfe.extract()
    return pd.Series(valores, index=nomes, dtype=float)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    g1 = mf_nativo(linha2, "gray")
    g2 = mf_nativo(linha2, "gray")
    g3 = mf_nativo(linha2, "gray", renomear=True)
    oh = mf_nativo(linha2, "one-hot")

print("controle (gray vs gray)     :", mudou(g1, g2).sum(), "de", len(g1))
print("renomeado (gray vs gray')   :", mudou(g1, g3).sum(), "de", len(g1))
print("gray vs one-hot             :", mudou(g1, oh).sum(), "de", len(g1))
print("gray vs ordinal (a_)        :", mudou(g1, a_).sum(), "de", len(g1))
print(g1[mudou(g1, g3)].index.tolist())

controle (gray vs gray)     : 2 de 73
renomeado (gray vs gray')   : 27 de 73
gray vs one-hot             : 39 de 73
gray vs ordinal (a_)        : 27 de 73
['attr_conc.mean', 'attr_conc.sd', 'cor.mean', 'cor.sd', 'cov.mean', 'cov.sd', 'eigenvalues.mean', 'eigenvalues.sd', 'gravity', 'iq_range.mean', 'iq_range.sd', 'kurtosis.mean', 'kurtosis.sd', 'mean.mean', 'mean.sd', 'median.mean', 'median.sd', 'nr_cor_attr', 'nr_outliers', 'sd.mean', 'sd.sd', 'skewness.mean', 'skewness.sd', 't_mean.mean', 't_mean.sd', 'var.mean', 'var.sd']


In [9]:
com_cat = selected_tasks.NumberOfSymbolicFeatures > 1
print(com_cat.sum(), "de", len(selected_tasks), "bases têm atributos categóricos")
print(selected_tasks.loc[com_cat, ["name", "NumberOfSymbolicFeatures", "NumberOfNumericFeatures"]]
      .sort_values("NumberOfSymbolicFeatures", ascending=False).head(10))

rng = np.random.default_rng(0)

def mudou(x, z):
    return (x.isna() != z.isna()) | ((x - z).abs() > 1e-9)

def resumo(x, z, rotulo):
    m = mudou(x, z)
    ok = m & x.notna() & z.notna()
    rel = ((x - z).abs() / x.abs().clip(lower=1e-12))[ok]
    print(f"   {rotulo:<10}: {int(m.sum()):>2} de {len(x)} mudaram | "
          f"mudança relativa mediana {rel.median():.1%} | acima de 10%: {int((rel > 0.10).sum())}")

for tc in ["gray", "one-hot-full", None]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        u1  = mf_nativo(linha2, tc)
        u1b = mf_nativo(linha2, tc)                  
        u2  = mf_nativo(linha2, tc, renomear=True)
    print(f"\ntransform_cat = {tc}")
    resumo(u1, u1b, "controle")
    resumo(u1, u2, "renomeado")
    print(f"   NaN: {int(u1.isna().sum())} | infinitos: {int(np.isinf(u1).sum())}")
    print("   instáveis no controle:", u1[mudou(u1, u1b)].index.tolist())

22 de 60 bases têm atributos categóricos
                    name  NumberOfSymbolicFeatures  NumberOfNumericFeatures
167140               dna                       181                        0
45                splice                        61                        0
3               kr-vs-kp                        37                        0
14952   PhishingWebsites                        31                        0
9977               nomao                        30                       89
3021                sick                        23                        7
14954     cylinder-bands                        22                       18
31              credit-g                        14                        7
125920     dresses-sales                        12                        1
29       credit-approval                        10                        6

transform_cat = gray
   controle  :  2 de 73 mudaram | mudança relativa mediana 36.8% | acima de 10%: 2
   renomeado : 24 

In [10]:
for nome in ["kr-vs-kp", "splice", "PhishingWebsites", "credit-approval"]:
    lin = next(selected_tasks[selected_tasks.name == nome].itertuples())
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            t0 = time.time()
            u1 = mf_nativo(lin, "one-hot-full")
            seg = time.time() - t0
            u1b = mf_nativo(lin, "one-hot-full")
            u2 = mf_nativo(lin, "one-hot-full", renomear=True)
    except Exception as e:
        print(f"\n{nome}: ERRO {repr(e)[:200]}")
        continue
    print(f"\n{nome} ({seg:.1f}s)")
    resumo(u1, u1b, "controle")
    resumo(u1, u2, "renomeado")
    print(f"   NaN: {int(u1.isna().sum())} | infinitos: {int(np.isinf(u1).sum())}")
    print("   NaN em:", u1[u1.isna()].index.tolist())


kr-vs-kp (1.5s)
   controle  :  2 de 73 mudaram | mudança relativa mediana 94.8% | acima de 10%: 2
   renomeado :  2 de 73 mudaram | mudança relativa mediana 63.7% | acima de 10%: 2
   NaN: 3 | infinitos: 0
   NaN em: ['can_cor.sd', 'cat_to_num', 'sd_ratio']

splice (3.9s)
   controle  :  2 de 73 mudaram | mudança relativa mediana 62.7% | acima de 10%: 2
   renomeado :  2 de 73 mudaram | mudança relativa mediana 17.7% | acima de 10%: 2
   NaN: 2 | infinitos: 0
   NaN em: ['cat_to_num', 'sd_ratio']

PhishingWebsites (2.0s)
   controle  :  2 de 73 mudaram | mudança relativa mediana 14.8% | acima de 10%: 1
   renomeado :  2 de 73 mudaram | mudança relativa mediana 62.2% | acima de 10%: 2
   NaN: 3 | infinitos: 0
   NaN em: ['can_cor.sd', 'cat_to_num', 'sd_ratio']

credit-approval (1.3s)
   controle  :  2 de 73 mudaram | mudança relativa mediana 8.6% | acima de 10%: 0
   renomeado :  2 de 73 mudaram | mudança relativa mediana 11.3% | acima de 10%: 1
   NaN: 52 | infinitos: 0
   NaN em: ['

In [12]:
print((selected_tasks.NumberOfMissingValues > 0).sum(), "de", len(selected_tasks),
      "bases têm valores faltantes")

def mf_final(linha, transform_cat="one-hot-full"):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i, col in enumerate(X.columns):
        if i in cat_cols:
            s = X[col].astype(object)
            moda = s.mode(dropna=True)
            X[col] = s.fillna(moda.iloc[0] if len(moda) else "faltante")
        else:
            s = pd.to_numeric(X[col], errors="coerce")
            X[col] = s.fillna(s.median() if s.notna().any() else 0.0)
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=object), y_arr, cat_cols=cat_cols, transform_cat=transform_cat)
    nomes, valores = mfe.extract()
    return pd.Series(valores, index=nomes, dtype=float)

for nome in ["credit-approval", "dna"]:
    lin = next(selected_tasks[selected_tasks.name == nome].itertuples())
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            t0 = time.time()
            u = mf_final(lin)
            seg = time.time() - t0
        print(f"\n{nome}: {seg:.1f}s | NaN: {int(u.isna().sum())} | infinitos: {int(np.isinf(u).sum())}")
        print("   NaN em:", u[u.isna()].index.tolist())
    except Exception as e:
        print(f"\n{nome}: ERRO {repr(e)[:200]}")

9 de 60 bases têm valores faltantes

credit-approval: 0.9s | NaN: 2 | infinitos: 0
   NaN em: ['can_cor.sd', 'sd_ratio']

dna: 8.3s | NaN: 2 | infinitos: 0
   NaN em: ['cat_to_num', 'sd_ratio']


In [13]:
#Diagnóstico das 9 bases com faltantes
com_missing = selected_tasks[selected_tasks.NumberOfMissingValues > 0].copy()
linhas = []

for row in com_missing.itertuples():
    X, y, cat_ind, _ = pegar_dados(row.did, row.target_feature)
    frac_total = X.isna().sum().sum() / X.size
    frac_col = X.isna().mean()
    cols_miss = frac_col[frac_col > 0]
    cat_mask = pd.Series(cat_ind, index=X.columns)

    n_num_miss = int((~cat_mask & (frac_col > 0)).sum())
    n_cat_miss = int((cat_mask & (frac_col > 0)).sum())
    max_col_miss = float(cols_miss.max()) if len(cols_miss) else 0.0
    linhas_com_miss = float(X.isna().any(axis=1).mean())

    linhas.append({
        "name": row.name,
        "instancias": len(X),
        "features": X.shape[1],
        "frac_missing": frac_total,
        "frac_linhas_com_miss": linhas_com_miss,
        "max_miss_numa_col": max_col_miss,
        "cols_num_miss": n_num_miss,
        "cols_cat_miss": n_cat_miss,
    })

df_miss = pd.DataFrame(linhas).sort_values("frac_missing", ascending=False)
display(df_miss.style.format({
    "frac_missing": "{:.2%}",
    "frac_linhas_com_miss": "{:.1%}",
    "max_miss_numa_col": "{:.1%}",
}))

,name,instancias,features,frac_missing,frac_linhas_com_miss,max_miss_numa_col,cols_num_miss,cols_cat_miss
7,dresses-sales,500,12,13.92%,80.2%,53.2%,0,9
3,sick,3772,29,5.54%,100.0%,100.0%,7,1
6,cylinder-bands,540,37,5.00%,48.7%,28.9%,16,12
2,eucalyptus,736,19,3.20%,12.9%,12.8%,9,0
8,MiceProtein,1080,77,1.68%,48.9%,26.4%,49,0
5,adult,48842,14,0.95%,7.4%,5.8%,0,3
1,credit-approval,690,15,0.65%,5.4%,1.9%,2,5
0,breast-w,699,9,0.25%,2.3%,2.3%,1,0
4,jm1,10885,21,0.01%,0.0%,0.0%,5,0


In [14]:
#Impacto da moda na imputação (possível pico artifical nas categorias)
def extrair_com_estrategia(linha, cat_strategy="moda"):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]

    for i, col in enumerate(X.columns):
        if i in cat_cols:
            s = X[col].astype(object)
            if cat_strategy == "moda":
                moda = s.mode(dropna=True)
                X[col] = s.fillna(moda.iloc[0] if len(moda) else "__MISSING__")
            else:
                X[col] = s.fillna("__MISSING__")
        else:
            s = pd.to_numeric(X[col], errors="coerce")
            X[col] = s.fillna(s.median() if s.notna().any() else 0.0)

    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=object), y_arr, cat_cols=cat_cols, transform_cat="one-hot-full")
    nomes, valores = mfe.extract()
    s_mf = pd.Series(valores, index=nomes, dtype=float)
    return s_mf.drop(labels=[c for c in ["attr_conc.mean", "attr_conc.sd"] if c in s_mf.index])

bases_cat_miss = df_miss[df_miss.cols_cat_miss > 0]["name"].tolist()
for nome in bases_cat_miss:
    lin = next(selected_tasks[selected_tasks.name == nome].itertuples())
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        s_moda = extrair_com_estrategia(lin, "moda")
        s_cat  = extrair_com_estrategia(lin, "missing")

    m = mudou(s_moda, s_cat)
    ok = m & s_moda.notna() & s_cat.notna()
    rel = ((s_moda - s_cat).abs() / s_moda.abs().clip(lower=1e-12))[ok].sort_values(ascending=False)

    print(f"\n{nome}: {int(m.sum())}/{len(s_moda)} mudaram | >10%: {int((rel > 0.10).sum())} | NaN moda={int(s_moda.isna().sum())}, __MISSING__={int(s_cat.isna().sum())}")
    if len(rel):
        print("   top 5 maiores diferenças relativas:")
        for k, v in rel.head(5).items():
            print(f"     {k:<20} moda={s_moda[k]:.4g} | __MISSING__={s_cat[k]:.4g} ({v:.1%})")


dresses-sales: 48/71 mudaram | >10%: 10 | NaN moda=2, __MISSING__=2
   top 5 maiores diferenças relativas:
     median.mean          moda=0.07436 | __MISSING__=0.05212 (29.9%)
     iq_range.mean        moda=0.07756 | __MISSING__=0.09758 (25.8%)
     joint_ent.sd         moda=0.4905 | __MISSING__=0.3933 (19.8%)
     sparsity.sd          moda=0.0619 | __MISSING__=0.04975 (19.6%)
     attr_ent.sd          moda=0.4975 | __MISSING__=0.4025 (19.1%)

sick: 49/71 mudaram | >10%: 0 | NaN moda=8, __MISSING__=8
   top 5 maiores diferenças relativas:
     nr_bin               moda=20 | __MISSING__=19 (5.0%)
     nr_cor_attr          moda=0.03556 | __MISSING__=0.03424 (3.7%)
     cov.mean             moda=1.048 | __MISSING__=1.009 (3.7%)
     nr_outliers          moda=47 | __MISSING__=48 (2.1%)
     t_mean.mean          moda=5.487 | __MISSING__=5.384 (1.9%)

cylinder-bands: 49/71 mudaram | >10%: 10 | NaN moda=8, __MISSING__=8
   top 5 maiores diferenças relativas:
     nr_bin               moda=6 

In [ ]:
#Percas de colinearidade com one-hot-full
com_cat_tasks = selected_tasks[selected_tasks.NumberOfSymbolicFeatures > 1].copy()
res_oh = []

for lin in com_cat_tasks.itertuples():
    X, y, cat_ind, _ = pegar_dados(lin.did, lin.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]

    for i, col in enumerate(X.columns):
        if i in cat_cols:
            s = X[col].astype(object)
            moda = s.mode(dropna=True)
            X[col] = s.fillna(moda.iloc[0] if len(moda) else "__MISSING__")
        else:
            s = pd.to_numeric(X[col], errors="coerce")
            X[col] = s.fillna(s.median() if s.notna().any() else 0.0)

    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    n_orig = X.shape[1]
    n_num = n_orig - len(cat_cols)
    card_total = sum(X.iloc[:, i].nunique() for i in cat_cols)
    n_pos_oh_full = n_num + card_total

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mfe_full = MFE(groups=["general", "statistical", "info-theory"])
        mfe_full.fit(X.to_numpy(dtype=object), y_arr, cat_cols=cat_cols, transform_cat="one-hot-full")
        nf, vf = mfe_full.extract()
        sf = pd.Series(vf, index=nf, dtype=float).drop(labels=["attr_conc.mean", "attr_conc.sd"], errors="ignore")

        try:
            mfe_k1 = MFE(groups=["general", "statistical", "info-theory"])
            mfe_k1.fit(X.to_numpy(dtype=object), y_arr, cat_cols=cat_cols, transform_cat="one-hot")
            nk, vk = mfe_k1.extract()
            sk = pd.Series(vk, index=nk, dtype=float).drop(labels=["attr_conc.mean", "attr_conc.sd"], errors="ignore")
            nan_k1 = int(sk.isna().sum())
            nan_so_full = sorted(sf[sf.isna() & sk.notna()].index.tolist())
            nan_so_k1 = sorted(sk[sk.isna() & sf.notna()].index.tolist())
        except ValueError:
            nan_k1 = "ERRO (col constante)"
            nan_so_full = ["k-1 falhou"]
            nan_so_k1 = []

    res_oh.append({
        "name": lin.name,
        "cols_orig": n_orig,
        "cols_oh_full": n_pos_oh_full,
        "fator_expansao": round(n_pos_oh_full / n_orig, 2),
        "nan_full": int(sf.isna().sum()),
        "nan_k1": nan_k1,
        "nan_so_no_full": nan_so_full,
        "nan_so_no_k1": nan_so_k1,
    })

df_oh = pd.DataFrame(res_oh).sort_values("fator_expansao", ascending=False)
display(df_oh)

ValueError: This type of one-hot encoding does not support features with 1 or less distinct values. Drop the 21th categorical feature or select another encoding strategy.